<a href="https://colab.research.google.com/github/Rei-stark/concursos/blob/main/Experi%C3%AAncia_MINST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# 1. Carregar a base de dados MNIST (dígitos escritos à mão)
print("A carregar os dados MNIST...")
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train, -1).astype("float32") / 255
x_test = np.expand_dims(x_test, -1).astype("float32") / 255

# 2. Definir a dimensão do Espaço Latente (2 dimensões para podermos visualizar num gráfico X,Y)
latent_dim = 2

# --- CONSTRUÇÃO DO ENCODER (Comprime a imagem) ---
encoder_inputs = keras.Input(shape=(28, 28, 1))
x = layers.Flatten()(encoder_inputs)
x = layers.Dense(256, activation="relu")(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)

# Função de Amostragem (O truque estatístico do VAE que permite a geração)
def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.keras.backend.random_normal(shape=tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = layers.Lambda(sampling, name="z")([z_mean, z_log_var])
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

# --- CONSTRUÇÃO DO DECODER (Gera a imagem) ---
latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(256, activation="relu")(latent_inputs)
x = layers.Dense(28 * 28, activation="sigmoid")(x)
decoder_outputs = layers.Reshape((28, 28, 1))(x)
decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")

# --- CLASSE DO MODELO VAE (Juntando tudo com a matemática do Erro/Loss) ---
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            # Perda de Reconstrução (A imagem gerada parece real?)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2))
            )
            # Perda KL (O espaço latente está bem distribuído?)
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        return {"perda_total": total_loss}

print("✅ Arquitetura VAE construída com sucesso!")

A carregar os dados MNIST...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
✅ Arquitetura VAE construída com sucesso!


In [ ]:
# Instanciar e compilar o modelo
vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001))

print("Iniciando o treino da Inteligência Artificial Generativa...")
print("Isto levará cerca de 1 a 2 minutos com a aceleração GPU ativada.\n")

# Iniciar o treino
history = vae.fit(x_train, epochs=10, batch_size=128)

print("\n✅ Treino concluído! O modelo aprendeu a representação latente dos números.")

Iniciando o treino da Inteligência Artificial Generativa...
Isto levará cerca de 1 a 2 minutos com a aceleração GPU ativada.

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - perda_total: 175.1849
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - perda_total: 167.6165
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - perda_total: 151.9311
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - perda_total: 166.7022
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - perda_total: 167.7789
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - perda_total: 157.0048
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - perda_total: 160.7399
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - perda_total: 162.1653
Epoch 9/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - perda_total: 157.5044
Epoch 10/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - perda_total: 161.4044

✅ Treino concluído! O modelo aprendeu a representação latente dos números.


In [ ]:
def gerar_digito_interativo(x, y):
    # Recebemos as coordenadas X e Y dos sliders (o Espaço Latente)
    z_sample = np.array([[x, y]])

    # Pedimos ao DECODER para gerar uma imagem a partir desta coordenada estatística
    x_decoded = decoder.predict(z_sample, verbose=0)
    digit = x_decoded[0].reshape(28, 28)

    # Visualização
    plt.figure(figsize=(4, 4))
    plt.imshow(digit, cmap='Greys_r')
    plt.title(f"Coordenada do Espaço Latente: X={x}, Y={y}", fontsize=10)
    plt.axis('off')
    plt.show()

print("--- LABORATÓRIO DE IA GENERATIVA ---")
print("Mexa nos sliders abaixo para navegar no 'cérebro' da máquina.")
print("Observe como a máquina não está a buscar uma imagem pronta, mas a GERAR um traço novo em tempo real!")

# Criar a interface interativa com ipywidgets
widgets.interact(gerar_digito_interativo,
                 x=widgets.FloatSlider(value=0, min=-3, max=3, step=0.1, description='Eixo X:', continuous_update=False),
                 y=widgets.FloatSlider(value=0, min=-3, max=3, step=0.1, description='Eixo Y:', continuous_update=False))

--- LABORATÓRIO DE IA GENERATIVA ---
Mexa nos sliders abaixo para navegar no 'cérebro' da máquina.
Observe como a máquina não está a buscar uma imagem pronta, mas a GERAR um traço novo em tempo real!


interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='Eixo X:', max=3.0, min=-3.0…

<function __main__.gerar_digito_interativo(x, y)>